# PELIC — Load & Filter by L1
This notebook clones the PELIC dataset, inspects L1 coverage, and exports per-L1 text files ready for stylometric feature extraction.

## 1. Clone the repo (run once)

In [ ]:
import os

if not os.path.exists("PELIC-dataset"):
    os.system("git clone https://github.com/ELI-Data-Mining-Group/PELIC-dataset")
    print("Cloned.")
else:
    print("Already cloned.")

## 2. Load the compiled CSV

In [ ]:
import pandas as pd

df = pd.read_csv("PELIC-dataset/PELIC_compiled.csv")
print(f"Total rows: {len(df):,}")
print(f"Columns: {df.columns.tolist()}")
df.head(3)

## 3. Inspect L1 coverage
See which L1s are available and how many texts each has.

In [ ]:
l1_counts = df["L1"].value_counts()
print(f"Total unique L1s: {len(l1_counts)}\n")
print(l1_counts.to_string())

## 4. Configure target L1s
Edit this cell to match exactly what the L1 column uses (check the output above for exact spelling).

In [ ]:
# Adjust these strings to match the exact L1 labels in the corpus
TARGET_L1S = {
    "spanish": "Spanish",
    "french":  "French",
    "italian": "Italian",
}

# Text column — PELIC uses 'text' but confirm from your column list above
TEXT_COL = "text"

# Minimum token count per text — filters out very short submissions
MIN_TOKENS = 50

## 5. Filter, clean, and report

In [ ]:
def basic_clean(text):
    """Minimal cleaning: strip whitespace, drop empty."""
    if not isinstance(text, str):
        return None
    text = text.strip()
    return text if len(text) > 0 else None

subsets = {}

for label, l1_value in TARGET_L1S.items():
    subset = df[df["L1"] == l1_value][["L1", TEXT_COL]].copy()
    subset[TEXT_COL] = subset[TEXT_COL].apply(basic_clean)
    subset = subset.dropna(subset=[TEXT_COL])

    # Filter by minimum token count
    subset = subset[subset[TEXT_COL].str.split().str.len() >= MIN_TOKENS]

    subsets[label] = subset
    total_words = subset[TEXT_COL].str.split().str.len().sum()
    print(f"{label:10s} | texts: {len(subset):>5,} | total words: {total_words:>9,}")

## 6. Inspect samples

In [ ]:
for label, subset in subsets.items():
    print(f"\n--- {label.upper()} sample ---")
    print(subset[TEXT_COL].iloc[0][:400])
    print("...")

## 7. Export per-L1 text files
Saves one `.txt` file per L1 (one text per line) and one combined `.csv` with an `l1` column — both formats useful downstream.

In [ ]:
import os

os.makedirs("pelic_filtered", exist_ok=True)

all_frames = []

for label, subset in subsets.items():
    # One text per line .txt
    out_txt = f"pelic_filtered/{label}.txt"
    with open(out_txt, "w", encoding="utf-8") as f:
        for text in subset[TEXT_COL]:
            f.write(text.replace("\n", " ") + "\n")
    print(f"Saved {out_txt}")

    # Add l1 label for combined CSV
    tagged = subset[[TEXT_COL]].copy()
    tagged["l1"] = label
    all_frames.append(tagged)

# Combined CSV
combined = pd.concat(all_frames, ignore_index=True)
combined.to_csv("pelic_filtered/pelic_l1_combined.csv", index=False)
print(f"\nSaved combined CSV: {len(combined):,} texts across {combined['l1'].nunique()} L1s")

## 8. Coverage warning
If French or Italian have very few texts, note it here for your methodology section.

In [ ]:
MINIMUM_VIABLE = 200  # texts — below this, the L1 profile may be unstable

for label, subset in subsets.items():
    n = len(subset)
    if n < MINIMUM_VIABLE:
        print(f"WARNING: {label} has only {n} texts (< {MINIMUM_VIABLE}). "
              f"Consider supplementing with EFCAMDAT.")
    else:
        print(f"OK: {label} has {n:,} texts — sufficient for stable stylometric profiles.")